# Avro Basics — 05: Nullable Unions


Avro represents nullable fields as unions with null.
This is the most common pattern you'll encounter in Avro schemas.

Topics: ["null","T"] pattern, default values for unions, complex unions,
reading legacy Avro with nullable fields, null handling in Spark.


In [1]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 13:10:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Avro JAR: spark-avro_2.13-4.0.2


## Step 1 — The Nullable Union Pattern

Nullable fields use union with null. Order matters for the default.

In [2]:

import json as pyjson

# WRONG: no default — required field
# CORRECT nullable patterns
print("Nullable union patterns:")
print()
print('  ["null","string"]  — nullable, default null:')
print('  {"type":["null","string"],"default":null}')
print()
print('  ["string","null"]  — nullable, default "" (string first):')
print('  {"type":["string","null"],"default":""}')
print()
print("Rule: the FIRST type in the union must match the default value type")

NULLABLE_SCHEMA = pyjson.dumps({"type":"record","name":"Customer","fields":[
    {"name":"id",          "type":"long"},
    {"name":"name",        "type":"string"},
    {"name":"email",       "type":["null","string"],  "default":None},   # nullable, default null
    {"name":"phone",       "type":["string","null"],  "default":""},     # nullable, default ""
    {"name":"score",       "type":["null","double"],  "default":None},   # nullable double
    {"name":"is_vip",      "type":["null","boolean"], "default":None},   # nullable boolean
]})
print("\nSchema with nullable fields created ✅")


Nullable union patterns:

  ["null","string"]  — nullable, default null:
  {"type":["null","string"],"default":null}

  ["string","null"]  — nullable, default "" (string first):
  {"type":["string","null"],"default":""}

Rule: the FIRST type in the union must match the default value type

Schema with nullable fields created ✅


## Step 2 — Write and Read Nullable Data



In [3]:

customers = spark.createDataFrame([
    (1, "Alice", "alice@ex.com",  "+1-555-0101", 9.5,  True),
    (2, "Bob",   None,             "",            None, False),
    (3, "Carol", "carol@ex.com",  None,           7.2,  None),
], ["id","name","email","phone","score","is_vip"])

CUST_PATH = f"{DATA_DIR}/nullable_customers"
customers.write.format("avro").option("avroSchema", NULLABLE_SCHEMA) \
         .mode("overwrite").save(CUST_PATH)

df_cust = spark.read.format("avro").load(CUST_PATH)
print("Nullable fields in Avro:")
df_cust.printSchema()
print()
df_cust.show(truncate=False)
print("null values preserved correctly ✅")


Nullable fields in Avro:
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- score: double (nullable = true)
 |-- is_vip: boolean (nullable = true)


+---+-----+------------+-----------+-----+------+
|id |name |email       |phone      |score|is_vip|
+---+-----+------------+-----------+-----+------+
|1  |Alice|alice@ex.com|+1-555-0101|9.5  |true  |
|3  |Carol|carol@ex.com|NULL       |7.2  |NULL  |
|2  |Bob  |NULL        |           |NULL |false |
+---+-----+------------+-----------+-----+------+

null values preserved correctly ✅


## Step 3 — Handling Nulls in Queries



In [4]:

df_cust = spark.read.format("avro").load(f"{DATA_DIR}/nullable_customers")

print("Null count per column:")
df_cust.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_cust.columns]).show()

print("Fill nulls with defaults:")
df_filled = df_cust \
    .fillna({"email": "unknown@example.com", "phone": "N/A", "score": 0.0}) \
    .fillna({"is_vip": False})
df_filled.show(truncate=False)


Null count per column:
+---+----+-----+-----+-----+------+
| id|name|email|phone|score|is_vip|
+---+----+-----+-----+-----+------+
|  0|   0|    1|    1|    1|     1|
+---+----+-----+-----+-----+------+

Fill nulls with defaults:
+---+-----+-------------------+-----------+-----+------+
|id |name |email              |phone      |score|is_vip|
+---+-----+-------------------+-----------+-----+------+
|1  |Alice|alice@ex.com       |+1-555-0101|9.5  |true  |
|3  |Carol|carol@ex.com       |N/A        |7.2  |false |
|2  |Bob  |unknown@example.com|           |0.0  |false |
+---+-----+-------------------+-----------+-----+------+



## Step 4 — Complex Unions (Multiple Non-Null Types)



In [5]:

# Complex union: value can be int OR string OR null
# Less common but valid in Avro
COMPLEX_SCHEMA = pyjson.dumps({"type":"record","name":"Event","fields":[
    {"name":"event_id", "type":"long"},
    # metadata: can be a string or a number
    {"name":"metadata", "type":["null","string","long"], "default":None},
]})

complex_data = spark.createDataFrame([
    (1, None),
    (2, "campaign_abc"),
    (3, "42"),  # number as string (Spark maps complex union to string)
], ["event_id","metadata"])

COMP_PATH = f"{DATA_DIR}/complex_union"
try:
    complex_data.write.format("avro").option("avroSchema",COMPLEX_SCHEMA) \
                .mode("overwrite").save(COMP_PATH)
    print("Complex union written ✅")
    spark.read.format("avro").load(COMP_PATH).show()
except Exception as e:
    print(f"Complex union limitation: {type(e).__name__}: {str(e)[:100]}")
    print("Recommendation: avoid multi-non-null unions — use separate typed fields instead")


26/04/09 13:10:38 WARN TaskSetManager: Lost task 3.0 in stage 8.0 (TID 20) (172.18.0.5 executor 0): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to file:/workspace/data/avro_basics/complex_union/_temporary/0/_temporary/attempt_202604091310388709611004969234074_0008_m_000003_20/part-00003-6fd2ac29-d194-4e79-816a-50d1d5134802-c000.snappy.avro. SQLSTATE: 58030
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:772)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.enrichWriteError(FileFormatDataWriter.scala:92)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.writeWithMetrics(FileFormatDataWriter.scala:103)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.writeWithIterator(FileFormatDataWriter.scala:111)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$executeTask$1(FileFormatWriter.scala:406)
	at org.apache.s

Complex union limitation: Py4JJavaError: An error occurred while calling o168.save.
: org.apache.spark.SparkException: [TASK_WRITE_FAILED] Ta
Recommendation: avoid multi-non-null unions — use separate typed fields instead


## Summary



In [6]:

print("""
Nullable field patterns:
  Required  : {"name":"col","type":"string"}
  Nullable  : {"name":"col","type":["null","string"],"default":null}
  Optional  : {"name":"col","type":["string","null"],"default":""}

Spark mapping:
  ["null","string"] → StringType(nullable=true)
  ["null","long"]   → LongType(nullable=true)
  ["null","double"] → DoubleType(nullable=true)

Always use ["null","T"] (null first) when the default is null.
Use ["T","null"] only when you want a non-null default.
""")



Nullable field patterns:
  Required  : {"name":"col","type":"string"}
  Nullable  : {"name":"col","type":["null","string"],"default":null}
  Optional  : {"name":"col","type":["string","null"],"default":""}

Spark mapping:
  ["null","string"] → StringType(nullable=true)
  ["null","long"]   → LongType(nullable=true)
  ["null","double"] → DoubleType(nullable=true)

Always use ["null","T"] (null first) when the default is null.
Use ["T","null"] only when you want a non-null default.

